In [50]:
import pandas as pd

In [51]:
loan_data = pd.read_csv("loan_data.csv", sep = '\t')

handel_missing_values

In [52]:
cat_cols = loan_data.select_dtypes(include = ["object"]).columns
num_cols = loan_data.select_dtypes(include = ["number"]).columns

In [53]:
from sklearn.impute import SimpleImputer
num_imp = SimpleImputer(strategy='mean')
loan_data[num_cols] = num_imp.fit_transform(loan_data[num_cols])

In [54]:
cat_imp = SimpleImputer(strategy='most_frequent')
loan_data[cat_cols] = cat_imp.fit_transform(loan_data[cat_cols])

EDA

In [55]:
loan_data.drop(["Applicant_ID"], axis = 1)

,Applicant_Income,Coapplicant_Income,Employment_Status,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,Loan_Term,Loan_Purpose,Property_Area,Education_Level,Gender,Employer_Category,Loan_Approved
0,17795.000000,1387.0,Salaried,51.0,Married,0.0,637.0,4.0,0.53,19403.000000,45638.0,16619.0,84.0,Personal,Urban,Not Graduate,Female,Private,No
1,2860.000000,2679.0,Salaried,46.0,Married,3.0,621.0,2.0,0.30,2580.000000,49272.0,38687.0,48.0,Car,Semiurban,Graduate,Male,Private,No
2,7390.000000,2106.0,Salaried,25.0,Single,2.0,674.0,4.0,0.20,13844.000000,6908.0,27943.0,72.0,Business,Urban,Graduate,Female,Government,Yes
3,13964.000000,8173.0,Salaried,40.0,Married,2.0,579.0,3.0,0.31,9553.000000,10844.0,27819.0,60.0,Business,Rural,Graduate,Female,Government,No
4,13284.000000,4223.0,Self-employed,31.0,Single,2.0,721.0,1.0,0.29,9386.000000,37629.0,12741.0,72.0,Car,Urban,Graduate,Male,Private,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,10852.571579,9092.0,Salaried,58.0,Married,0.0,557.0,0.0,0.59,5370.000000,43563.0,8311.0,72.0,Personal,Urban,Not Graduate,Male,Unemployed,No
996,3279.000000,6356.0,Self-employed,58.0,Married,1.0,646.0,3.0,0.19,9940.452632,18361.0,22563.0,12.0,Business,Urban,Graduate,Female,Government,No
997,15192.000000,8433.0,Contract,48.0,Single,1.0,666.0,1.0,0.40,8581.000000,41335.0,16203.0,24.0,Home,Rural,Graduate,Male,MNC,No
998,9083.000000,7380.0,Unemployed,50.0,Single,1.0,748.0,3.0,0.31,13491.000000,8933.0,10290.0,36.0,Personal,Urban,Graduate,Male,Private,Yes


In [56]:
cat_cols

Index(['Employment_Status', 'Marital_Status', 'Loan_Purpose', 'Property_Area',
       'Education_Level', 'Gender', 'Employer_Category', 'Loan_Approved'],
      dtype='object')

Encoding 

In [61]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
le = LabelEncoder()
loan_data["Marital_Status"] = le.fit_transform(loan_data["Marital_Status"])
loan_data["Education_Level"] = le.fit_transform(loan_data["Education_Level"])
loan_data["Gender"] = le.fit_transform(loan_data["Gender"])
loan_data["Loan_Approved"] = le.fit_transform(loan_data["Loan_Approved"])

enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

oneHot = ["Employment_Status", "Loan_Purpose",
          "Property_Area", "Employer_Category"]

cat_data = enc.fit_transform(loan_data[oneHot])

cat_df = pd.DataFrame(
    cat_data,
    columns=enc.get_feature_names_out(oneHot),
    index=loan_data.index
)

loan_data = loan_data.drop(columns=oneHot)
loan_data = pd.concat([loan_data, cat_df], axis=1)

In [63]:
loan_data.head()

,Applicant_ID,Applicant_Income,Coapplicant_Income,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,...,Loan_Purpose_Home,Loan_Purpose_Personal,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban,Employer_Category_Business,Employer_Category_Government,Employer_Category_MNC,Employer_Category_Private,Employer_Category_Unemployed
0,1.0,17795.0,1387.0,51.0,0,0.0,637.0,4.0,0.53,19403.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,2.0,2860.0,2679.0,46.0,0,3.0,621.0,2.0,0.30,2580.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3.0,7390.0,2106.0,25.0,1,2.0,674.0,4.0,0.20,13844.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,4.0,13964.0,8173.0,40.0,0,2.0,579.0,3.0,0.31,9553.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,5.0,13284.0,4223.0,31.0,1,2.0,721.0,1.0,0.29,9386.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


logistic_reg

In [64]:
X = loan_data.drop(columns = ["Loan_Approved"])
y = loan_data["Loan_Approved"]

In [65]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

feature_scalling

In [72]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [73]:
from sklearn.linear_model import LogisticRegression
regModel = LogisticRegression()
regModel.fit(x_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [74]:
from sklearn.metrics import precision_score
y_pred = regModel.predict(x_test_scaled)
precision =  precision_score(y_test, y_pred)
print(precision)

0.7868852459016393


naive_bayes_algo

In [75]:
X = loan_data.drop(columns = ["Loan_Approved"])
y = loan_data["Loan_Approved"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(x_train_scaled, y_train)

,priors,None
,var_smoothing,1e-09


In [76]:
from sklearn.metrics import precision_score
y_pred = gnb.predict(x_test_scaled)
precision =  precision_score(y_test, y_pred)
print(precision)

0.6440677966101694
